# Giai đoạn 1: Chuẩn bị và Chuyển đổi Dữ liệu (Data Conversion)
Notebook này thực hiện các tác vụ:
1. Convert label của tập **ExDark** sang chuẩn YOLO (kèm xử lý tọa độ tràn viền).
2. Áp dụng tiền xử lý **CLAHE + Median Filter** để tạo ra tập baseline truyền thống.

**⚠️ YÊU CẦU TRƯỚC KHI CHẠY CHƯƠNG TRÌNH NÀY:**
Ngài hãy tải và chép dữ liệu raw của các tổ chức vào đúng cấu trúc sau:
- `data/raw/exdark/images/`
- `data/raw/exdark/labels/` (Chứa các file txt gốc)
- `data/raw/bdd100k_night/images/`
- `data/raw/bdd100k_night/labels/bdd100k_labels_images_train.json`

In [2]:
# Cài đặt các thư viện lõi trước khi chạy toàn bộ Pipeline
import sys
!{sys.executable} -m pip install opencv-python "numpy<2.0.0" tqdm ijson


   ---------------------------------------- 2/2 [ijson]



In [6]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import shutil
import random

# Fix random seed để chia split luôn cố định
random.seed(42)

BASE_DIR = Path(os.getcwd()).parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
CLAHE_DIR = BASE_DIR / 'data' / 'clahe_baseline'

# Khởi tạo các thư mục đầu ra
for split in ['train', 'val']:
    (PROCESSED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
    (CLAHE_DIR / split / 'images').mkdir(parents=True, exist_ok=True)

print(" Đã khởi tạo cấu trúc thư mục đầu ra!")

 Đã khởi tạo cấu trúc thư mục đầu ra!


## 1. Xử lý tập ExDark (12 Classes)
- **Rủi ro đã xử lý**: Tọa độ của ExDark là 1-indexed. Có chứa header rác `% bbGt`. Có thể bị tràn viền ảnh.
- **Split**: 80% Train - 20% Val.

In [7]:
EXDARK_CLASSES = ['bicycle', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cup', 'dog', 'motorbike', 'people', 'table']
EXDARK_CLASS_MAP = {name: i for i, name in enumerate(EXDARK_CLASSES)}

def convert_exdark():
    raw_img_dir = RAW_DIR / 'exdark' / 'images'
    raw_label_dir = RAW_DIR / 'exdark' / 'labels'
    
    img_paths = list(raw_img_dir.rglob('*.*'))
    if not img_paths:
        print(" Chưa có data ExDark. hãy copy data vào thư mục raw")
        return
        
    print(f"Bắt đầu convert {len(img_paths)} ảnh ExDark...")
    
    random.shuffle(img_paths)
    split_idx = int(len(img_paths) * 0.8)
    train_paths = img_paths[:split_idx]
    val_paths = img_paths[split_idx:]
    
    def process_split(paths, split_name):
        count = 0
        for img_path in tqdm(paths, desc=f"ExDark {split_name}"):
            img = cv2.imread(str(img_path))
            if img is None: continue
            h_img, w_img, _ = img.shape
            
            rel_path = img_path.relative_to(raw_img_dir)
            label_path = raw_label_dir / rel_path.parent / (img_path.name + '.txt')
            if not label_path.exists():
                label_path = raw_label_dir / rel_path.parent / (img_path.stem + '.txt')
                if not label_path.exists(): continue
                
            yolo_labels = []
            with open(label_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith('%'): continue
                    
                    parts = line.split()
                    if len(parts) < 5: continue
                    
                    class_name = parts[0].lower()
                    if class_name not in EXDARK_CLASS_MAP: continue
                    class_id = EXDARK_CLASS_MAP[class_name]
                    
                    l = max(0.0, float(parts[1]) - 1.0)
                    t = max(0.0, float(parts[2]) - 1.0)
                    w = float(parts[3])
                    h = float(parts[4])
                    
                    x_center = np.clip((l + w / 2) / w_img, 0, 1)
                    y_center = np.clip((t + h / 2) / h_img, 0, 1)
                    w_norm = np.clip(w / w_img, 0, 1)
                    h_norm = np.clip(h / h_img, 0, 1)
                    
                    if w_norm == 0 or h_norm == 0: continue
                    yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
            
            if yolo_labels:
                out_img_path = PROCESSED_DIR / split_name / 'images' / img_path.name
                out_label_path = PROCESSED_DIR / split_name / 'labels' / (img_path.stem + '.txt')
                shutil.copy(img_path, out_img_path)
                with open(out_label_path, 'w', encoding='utf-8') as f:
                    f.writelines(yolo_labels)
                count += 1
        print(f" Đã xử lý {count} ảnh cho tập {split_name}.")

    process_split(train_paths, 'train')
    process_split(val_paths, 'val')

# Bỏ comment dòng dưới để chạy
convert_exdark()

Bắt đầu convert 7363 ảnh ExDark...


ExDark train: 100%|██████████| 5890/5890 [00:40<00:00, 145.90it/s]


 Đã xử lý 5890 ảnh cho tập train.


ExDark val: 100%|██████████| 1473/1473 [00:11<00:00, 131.60it/s]

 Đã xử lý 1472 ảnh cho tập val.


## 2. Tiền xử lý Baseline Truyền thống (CLAHE + Median Filter)
Tạo ra một tập dữ liệu sáng tự nhiên bằng thuật toán truyền thống để làm cột mốc (baseline) so sánh xem GAN có thực sự vượt trội hay không.

In [8]:
def preprocess_clahe(img_path, out_path):
    img = cv2.imread(str(img_path))
    if img is None: return
    
    # Khử nhiễu ISO ban đêm bằng Median Blur
    img = cv2.medianBlur(img, 3)
    
    # Áp dụng CLAHE trên kênh L (Lightness)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    cv2.imwrite(str(out_path), final)

def create_clahe_dataset():
    print("Bắt đầu sinh ảnh CLAHE Baseline...")
    for split in ['train', 'val']:
        in_dir = PROCESSED_DIR / split / 'images'
        out_dir = CLAHE_DIR / split / 'images'
        
        img_paths = list(in_dir.glob('*.*'))
        for img_path in tqdm(img_paths, desc=f"CLAHE {split}"):
            out_path = out_dir / img_path.name
            preprocess_clahe(img_path, out_path)
            
            in_label = PROCESSED_DIR / split / 'labels' / (img_path.stem + '.txt')
            out_label = CLAHE_DIR / split / 'labels' / (img_path.stem + '.txt')
            (CLAHE_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
            if in_label.exists():
                shutil.copy(in_label, out_label)
    print(" Hoàn tất CLAHE!")

# Bỏ comment dòng dưới để chạy
create_clahe_dataset()

Bắt đầu sinh ảnh CLAHE Baseline...


CLAHE val: 100%|██████████| 1472/1472 [00:23<00:00, 61.80it/s]

 Hoàn tất CLAHE!
